In [1]:
import joblib
import numpy as np 
import re
from sklearn.metrics.pairwise import cosine_similarity

def tokenize_ingredients(ingredient_list):
    text = " ".join(ingredient_list).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

print("loading successfull")

loading successfull


In [3]:
print("กำลังโหลดโมเดลจากโฟลเดอร์ models/ ...")

word2vec_model = joblib.load('../models/word2vec_model.joblib')
recipe_vectors = joblib.load('../models/recipe_vectors.joblib')

bigram_model = joblib.load('../models/bigram_model.joblib')

df_api = joblib.load('../models/df_for_api.joblib')

print("โหลดโมเดลทั้งหมดสำเร็จ! พร้อมใช้งาน")

กำลังโหลดโมเดลจากโฟลเดอร์ models/ ...
โหลดโมเดลทั้งหมดสำเร็จ! พร้อมใช้งาน


In [5]:
def get_recipe_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

def recommend_recipes(user_input, top_k=5):
    # 1. เตรียมคำและเข้า Bigram
    raw_tokens = tokenize_ingredients([user_input])
    user_tokens = bigram_model[raw_tokens]
    
    # 2. แปลงเป็น Vector
    user_vec = get_recipe_vector(user_tokens, word2vec_model).reshape(1, -1)
    if np.all(user_vec == 0):
        return {"error": "ไม่รู้จักวัตถุดิบนี้เลย"}
    
    # 3. คำนวณความเหมือน
    similarities = cosine_similarity(user_vec, recipe_vectors)[0]
    top_indices = similarities.argsort()[-top_k:][::-1]
    
    # 4. ดึงข้อมูลเตรียมส่งกลับ (จำลองรูปแบบ JSON)
    results = []
    for idx in top_indices:
        row = df_api.iloc[idx]
        results.append({
            "name": row['name'],
            "category": row['category'],
            "ingredients": row['ingredients'],
            "match_score": float(similarities[idx]) # แปลงเป็น float ปกติให้ API อ่านง่าย
        })
    return results

print("ฟังก์ชัน Core พร้อมทำงาน")

ฟังก์ชัน Core พร้อมทำงาน


In [6]:
# จำลองแอปมือถือส่งคำว่า "garlic powder chicken" มาให้
mobile_app_request = "garlic powder chicken"
print(f"📱 Mobile App ส่งข้อมูลมาว่า: {mobile_app_request}\n")

# ประมวลผล
response_data = recommend_recipes(mobile_app_request, top_k=3)

# แสดงผลลัพธ์แบบ JSON
import json
print(json.dumps(response_data, indent=2, ensure_ascii=False))

📱 Mobile App ส่งข้อมูลมาว่า: garlic powder chicken

[
  {
    "name": "easy baked tenders",
    "category": "Dessert",
    "ingredients": [
      "chicken tenders",
      "butter",
      "season salt",
      "garlic powder"
    ],
    "match_score": 0.8712434768676758
  },
  {
    "name": "baked chili chicken",
    "category": "Salad",
    "ingredients": [
      "chicken coating mix",
      "chili powder",
      "garlic powder",
      "boneless skinless chicken breasts"
    ],
    "match_score": 0.869538426399231
  },
  {
    "name": "ramen noodle chicken seasoning packet substitute",
    "category": "Pasta/Noodle",
    "ingredients": [
      "onion powder",
      "garlic powder",
      "poultry seasoning",
      "black pepper",
      "chicken"
    ],
    "match_score": 0.8655842542648315
  }
]
